# Healthy Start — Python Detection Server (Colab)

Runs the FastAPI dental detection server on Colab with a public tunnel.
The public URL connects to the Vercel frontend at `levelupwrestlingapp.com/hs/loop`.

**Requirements:** GPU runtime (T4 recommended). Go to Runtime > Change runtime type > T4 GPU.

## 1. Clone Repo & Install Dependencies

In [ ]:
# Mount Google Drive (for model persistence across restarts)
from google.colab import drive
drive.mount('/content/drive')

# Clone the repo (or pull latest if already cloned)
import os
if os.path.exists('/content/levelup'):
    !cd /content/levelup && git pull origin main
else:
    !git clone https://github.com/sportsmockery/LevelUp.git /content/levelup

%cd /content/levelup/python

# Install dependencies
!pip install -q ultralytics>=8.3.0 fastapi>=0.115.0 uvicorn>=0.34.0 \
  python-multipart>=0.0.18 Pillow>=11.0.0 \
  pyngrok torch torchvision opencv-python-headless roboflow

# segment-anything is not on PyPI — install from Meta's GitHub
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

print('\n--- Dependencies installed, Drive mounted ---')

## 2. Download Models

**SAM model** downloads automatically from Meta.  
**YOLO model** — upload `yolov12s_010826.pt` using the file panel on the left,  
or place it in Google Drive and mount below.

In [ ]:
import os, urllib.request, pathlib, shutil

MODEL_DIR = pathlib.Path('/content/levelup/python')
GDRIVE_HS = pathlib.Path('/content/drive/MyDrive/hs_models')
GDRIVE_HS.mkdir(parents=True, exist_ok=True)

# --- SAM vit_b (358 MB) ---
SAM_URL = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'
SAM_PATH = MODEL_DIR / 'sam_vit_b_01ec64.pth'

if not SAM_PATH.exists():
    # Try Drive first
    gdrive_sam = GDRIVE_HS / 'sam_vit_b_01ec64.pth'
    if gdrive_sam.exists():
        shutil.copy2(str(gdrive_sam), str(SAM_PATH))
        print(f'SAM restored from Drive ({SAM_PATH.stat().st_size / 1e6:.0f} MB)')
    else:
        print('Downloading SAM vit_b model (358 MB)...')
        urllib.request.urlretrieve(SAM_URL, str(SAM_PATH))
        # Save to Drive for next time
        shutil.copy2(str(SAM_PATH), str(gdrive_sam))
        print(f'SAM downloaded and saved to Drive')
else:
    print(f'SAM already exists ({SAM_PATH.stat().st_size / 1e6:.0f} MB)')

# --- YOLO base model (18 MB) ---
YOLO_PATH = MODEL_DIR / 'yolov12s_010826.pt'
if not YOLO_PATH.exists():
    gdrive_yolo = GDRIVE_HS / 'yolov12s_010826.pt'
    if gdrive_yolo.exists():
        shutil.copy2(str(gdrive_yolo), str(YOLO_PATH))
        print(f'YOLO base model restored from Drive ({YOLO_PATH.stat().st_size / 1e6:.0f} MB)')
    else:
        print('\n' + '='*60)
        print('Upload yolov12s_010826.pt to Files panel, then run cell 2b')
        print('='*60)
else:
    print(f'YOLO base model ready ({YOLO_PATH.stat().st_size / 1e6:.0f} MB)')
    # Save to Drive if not there
    gdrive_yolo = GDRIVE_HS / 'yolov12s_010826.pt'
    if not gdrive_yolo.exists():
        shutil.copy2(str(YOLO_PATH), str(gdrive_yolo))
        print('  (saved to Drive)')

# --- Restore trained models from Drive ---
det_local = MODEL_DIR / 'runs/detect/contacts_detector_v1/weights/best.pt'
cls_local = MODEL_DIR / 'runs/classify/contacts_classifier_v1/best.pt'
det_drive = GDRIVE_HS / 'detector_best.pt'
cls_drive = GDRIVE_HS / 'classifier_best.pt'

restored = []
if det_drive.exists() and not det_local.exists():
    det_local.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(det_drive), str(det_local))
    restored.append(f'Detector ({det_drive.stat().st_size / 1e6:.1f} MB)')

if cls_drive.exists() and not cls_local.exists():
    cls_local.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(cls_drive), str(cls_local))
    restored.append(f'Classifier ({cls_drive.stat().st_size / 1e6:.1f} MB)')

if restored:
    print(f'\nTrained models restored from Drive: {", ".join(restored)}')
    print('No retraining needed! Models persist across Colab restarts.')
else:
    if det_local.exists() and cls_local.exists():
        print('\nTrained models already present locally.')
    else:
        print('\nNo trained models found. Click Start on /hs/loop to auto-train (~20 min).')

print(f'\nDrive folder: /content/drive/MyDrive/hs_models/')
!ls -lh /content/drive/MyDrive/hs_models/ 2>/dev/null || echo '  (empty)'

## 2b. (Optional) Upload YOLO model manually

If the YOLO model wasn't found above, upload it with the Files panel then run this cell:

In [ ]:
# Run this after uploading yolov12s_010826.pt to /content/
import shutil, pathlib
src = pathlib.Path('/content/yolov12s_010826.pt')
dst = pathlib.Path('/content/levelup/python/yolov12s_010826.pt')
if src.exists() and not dst.exists():
    shutil.copy(str(src), str(dst))
    print(f'Copied to {dst}')
elif dst.exists():
    print('YOLO model already in place')
else:
    print('Upload yolov12s_010826.pt to /content/ first')

## 3. API Keys & Training Data

Set your Roboflow API key to auto-fetch dental datasets.  
Without this, the loop needs manually uploaded images.

In [ ]:
import os, pathlib

# --- Set API key (required for auto-fetch) ---
ROBOFLOW_API_KEY = ''  # <-- PASTE YOUR ROBOFLOW KEY HERE

if ROBOFLOW_API_KEY:
    os.environ['ROBOFLOW_API_KEY'] = ROBOFLOW_API_KEY
    print(f'ROBOFLOW_API_KEY set ({ROBOFLOW_API_KEY[:6]}...)')

    # Write .env.local so the server process picks it up at startup
    env_path = pathlib.Path('/content/levelup/.env.local')
    env_path.write_text(f'ROBOFLOW_API_KEY={ROBOFLOW_API_KEY}\n')
    print(f'Written to {env_path}')
else:
    print('WARNING: No ROBOFLOW_API_KEY set.')
    print('The loop will fail unless you manually upload images.\n')

# Create directories the server/loop expects
for d in ['data/raw/train/images', 'data/labeling_queue', 'data/truth_engine', 'data/audit_history', 'runs']:
    pathlib.Path(f'/content/levelup/python/{d}').mkdir(parents=True, exist_ok=True)
print('Data directories created')

# --- Pre-fetch datasets if key is available ---
if ROBOFLOW_API_KEY:
    print('\nFetching dental datasets from Roboflow...')
    os.environ['PYTHONPATH'] = '/content/levelup/python'
    import sys
    sys.path.insert(0, '/content/levelup/python')
    from broken_contacts.dataset_fetcher import fetch_all_datasets
    results = fetch_all_datasets(
        api_key=ROBOFLOW_API_KEY,
        target_dir='data/raw/train/images',
        skip_downloaded=True,
    )
    total = sum(r.get('images_copied', 0) for r in results)
    print(f'\nFetched {total} images across {len(results)} datasets')
    for r in results:
        print(f"  {r.get('dataset', '?')}: {r.get('images_copied', 0)} images")

# Count available images
img_dir = pathlib.Path('/content/levelup/python/data/raw/train/images')
img_count = len([f for f in img_dir.rglob('*') if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}])
print(f'\nTotal training images available: {img_count}')
if img_count == 0:
    print('WARNING: No images! Upload .jpg/.png files to data/raw/train/images/')

## 4. Start Public Tunnel (ngrok)

Get a free ngrok auth token at https://dashboard.ngrok.com/get-started/your-authtoken  
Paste it below. This gives you a public HTTPS URL that Vercel can reach.

In [ ]:
NGROK_AUTH_TOKEN = ''  # <-- PASTE YOUR NGROK TOKEN HERE

from pyngrok import ngrok

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
else:
    print('WARNING: No ngrok token. Get one free at https://dashboard.ngrok.com')
    print('Without a token, the tunnel may be rate-limited.')

# Open tunnel to port 8100
public_url = ngrok.connect(8100, 'http')
print('=' * 60)
print(f'PUBLIC URL: {public_url}')
print('=' * 60)
print(f'Set this in Vercel: HS_DETECTION_URL = {public_url}')

## 5. Start the Server

Run this cell to start the FastAPI server. Training happens automatically when you click Start on the loop page.
Visit levelupwrestlingapp.com/hs/loop and click **Start**.

In [ ]:
import os

# Set env vars so the shell subprocess can find modules and keys
os.environ['PYTHONPATH'] = '/content/levelup/python'

print('Starting FastAPI server on port 8100...')
print('Tunnel active — Vercel frontend can now connect.\n')

# Verify models exist before starting
for name in ['yolov12s_010826.pt', 'sam_vit_b_01ec64.pth']:
    path = f'/content/levelup/python/{name}'
    if os.path.exists(path):
        print(f'  {name}: OK ({os.path.getsize(path) / 1e6:.0f} MB)')
    else:
        print(f'  {name}: MISSING — server will fail to load!')

# Count images
import pathlib
img_dir = pathlib.Path('/content/levelup/python/data/raw/train/images')
img_count = len([f for f in img_dir.rglob('*') if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}])
print(f'  Training images: {img_count}')
print(f'  ROBOFLOW_API_KEY: {"set" if os.environ.get("ROBOFLOW_API_KEY") else "NOT SET"}')
print()

!cd /content/levelup/python && python -m uvicorn server:app --host 0.0.0.0 --port 8100

## 6. (Optional) Test the Server

In [ ]:
# Run in a separate cell while the server is running
# (open a new code cell, the server cell above will keep running)
import requests
r = requests.get('http://localhost:8100/health')
print(r.json())